# benchmarkoor-fetch quickstart

End-to-end walkthrough of the granular (Style B) API: resolve a suite, list runs, fetch raw test stats, join with the per-fixture trace, and inspect the result mid-pipeline.

If you just want the full artifact bundle on disk, prefer the one-shot `client.run(config)` form (see the README *Style A* example).

## Installing

In [1]:
!pip install benchmarkoor-fetch

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.5 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 10.6 MB/s  0:00:00 10.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 10.8 MB/s  0:00:00 10.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 10.8 MB/s  0:00:03 10.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [benchmarkoor-fetch]2m7/9 [pandas]pydantic]


## API key

The Benchmarkoor API requires a bearer token. Request one from the Benchmarkoor team (see [benchmarkoor-api.core.ethpandaops.io](https://benchmarkoor-api.core.ethpandaops.io)), then paste it into the cell below.

> **Warning:** Treat this token like a password. Do not commit it to git or share the notebook with the value filled in — clear the cell (or replace it with a placeholder) before checking the notebook in.

In [ ]:
BENCHMARKOOR_TOKEN = "paste-your-token-here"

In [3]:
from benchmarkoor_fetch import BenchmarkoorClient

client = BenchmarkoorClient(
    token=BENCHMARKOOR_TOKEN,
    cache_dir=".cache/benchmarkoor-fetch",
)

## 1. Resolve the latest suite

Given a `(network, fork, test_type)` tuple, the client returns the `suite_hash` of the most recent matching suite. Use `list_suites` if you want to see every match instead.

In [4]:
suite_hash = client.resolve_suite(
    network="jochemnet",
    fork="amsterdam",
    test_type="compute",
)
suite_hash

'3182dda7b93dee61'

## 2. List runs in a window

`start_date` is applied server-side; `end_date` and `run_id_pattern` are filtered in-process. `run_id_pattern` is a regex matched against each `run_id` with `re.fullmatch`.

In [5]:
runs = client.list_runs(
    suite_hash,
    start_date="2026-05-18",
    end_date="2026-05-20",
)
len(runs), runs[:1]

(26,
 [{'run_id': '1779301290_168ce940_nethermind-bal-full',
   'timestamp': 1779301290,
   'start_ts': '2026-05-20T18:21:30Z'}])

## 3. Fetch raw test stats

Paginated and threaded under the hood; pass `suite_hash` so per-run results are cached on disk.

In [6]:
raw_df = client.fetch_test_stats(runs, suite_hash=suite_hash)
raw_df.shape, raw_df.columns.tolist()

fetching test_stats (suite 3182dda7b9): 100%|██████████| 26/26 [00:20<00:00,  1.29it/s]


((126594, 5),
 ['client_name',
  'run_id',
  'ingestion_timestamp',
  'test_title',
  'test_runtime_ms'])

## 4. Fetch the trace summary

Per-fixture opcode counts, keyed by `test_title`.

In [7]:
trace = client.fetch_trace(suite_hash)
next(iter(trace.items()))

('test_account_query.py__test_codecopy_benchmark[fork_Amsterdam-benchmark_test-code_size_0-mem_size_0-benchmark_100M]',
 {'CODECOPY': 9985728.0,
  'DUP2': 1.0,
  'DUP3': 1.0,
  'JUMP': 1520.0,
  'JUMPDEST': 1526.0,
  'PUSH0': 19972975.0,
  'PUSH1': 9985735.0,
  'PUSH2': 1.0,
  'RETURN': 1.0})

## 5. Parse titles + join opcounts

Returns a `(bench_df, trace_df)` tuple. `bench_df` is the merged frame written to `bench_data.parquet`; `trace_df` is the per-fixture trace frame.

In [8]:
bench_df, trace_df = client.parse(raw_df, trace, fork="amsterdam")
bench_df.head()

,client_name,run_id,ingestion_timestamp,test_title,test_runtime_ms,test_file,test_name,test_params,test_opcode,block_limit_million,opcount
0,nethermind,1779301290_168ce940_nethermind-bal-full,2026-05-20 18:21:30+00:00,test_log.py__test_log_benchmark[fork_Amsterdam...,9.805041,test_log.py,test_log_benchmark,fork_Amsterdam-benchmark_test-log_size_1024-me...,LOG2,120,12852.0
1,nethermind,1779301290_168ce940_nethermind-bal-full,2026-05-20 18:21:30+00:00,test_call_context.py__test_calldataload[fork_A...,2.505748,test_call_context.py,test_calldataload,fork_Amsterdam-benchmark_test-zero_data_True-c...,CALLDATALOAD,100,6144.0
2,nethermind,1779301290_168ce940_nethermind-bal-full,2026-05-20 18:21:30+00:00,test_call_context.py__test_returndatacopy[fork...,103.111289,test_call_context.py,test_returndatacopy,fork_Amsterdam-benchmark_test-fixed_dst_True-r...,RETURNDATACOPY,240,23964100.0
3,nethermind,1779301290_168ce940_nethermind-bal-full,2026-05-20 18:21:30+00:00,test_account_query.py__test_codecopy_benchmark...,16.040407,test_account_query.py,test_codecopy_benchmark,fork_Amsterdam-benchmark_test-code_size_24576-...,CODECOPY,260,112196.0
4,nethermind,1779301290_168ce940_nethermind-bal-full,2026-05-20 18:21:30+00:00,test_arithmetic.py__test_arithmetic[fork_Amste...,170.465752,test_arithmetic.py,test_arithmetic,fork_Amsterdam-benchmark_test-opcode_MUL--benc...,MUL,260,32450576.0


Runtimes are in milliseconds (`test_runtime_ms`). Convert downstream if you need seconds.

## 6. Inspect unparsed fixtures

Rows whose `test_title` did not parse flow through with empty parsed columns. Surface them here instead of relying on the end-of-run warning.

In [9]:
unparsed = bench_df[bench_df["test_file"].isna()]
unparsed["test_title"].unique()[:10]

<ArrowStringArray>
[]
Length: 0, dtype: str